# AMPS I · Week 05 — HW5: Lax–Milgram and the method of manufactured solutions
**Due:** before the Week 6 lecture · **Points:** 100 · **Submit:** this notebook (`.ipynb`) with all outputs

**Rules**
- Before submitting: *Runtime → Restart session and run all*. Notebooks that do not run end to end receive 0.
- An error inside a `%%fenicsx` cell does **not** stop *Run all*. Read every output before you submit.
- LLM assistants are encouraged. Fill in the **AI-usage log** at the end.
- Every number you report must come from a cell you ran.

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## Q1 — Proofs (25 pts)
$\Omega\subset\mathbb R^2$ bounded, $V=H^1(\Omega)$ with $\|v\|^2=\|v\|^2_{L^2}+|v|^2_{H^1}$ unless stated otherwise.

**(a) (8)** Let $b(u,v)=\int_\Omega k\,\nabla u\cdot\nabla v+c\,uv$ with measurable coefficients $0<k_0\le k(x)\le k_1$, $0<c_0\le c(x)\le c_1$, and
$F(v)=\int_\Omega fv$. Show that $b$ is continuous and coercive on $V$ with explicit $M$ and $\gamma$, and state the Lax–Milgram bound for $\|u\|_{H^1}$.

**(b) (9)** Advection–diffusion on $V_0=\{v\in H^1(\Omega):v=0\text{ on }\partial\Omega\}$:
$a(u,v)=\int_\Omega\nabla u\cdot\nabla v+(\boldsymbol\beta\cdot\nabla u)\,v+c\,uv$ with a smooth velocity field $\boldsymbol\beta$ and a constant $c>0$.
Show that if $\nabla\cdot\boldsymbol\beta=0$ then $a(v,v)=|v|^2_{H^1}+c\|v\|^2_{L^2}$ for all $v\in V_0$, so $a$ is coercive although it is not symmetric.
What condition replaces $\nabla\cdot\boldsymbol\beta=0$ if $\nabla\cdot\boldsymbol\beta\ne0$?
*(Hint: $(\boldsymbol\beta\cdot\nabla v)v=\tfrac12\boldsymbol\beta\cdot\nabla(v^2)$, then the divergence theorem.)*

**(c) (8)** Let $b$ be symmetric, continuous ($M$) and coercive ($\gamma$) on $V$, $u$ the solution in $V$ and $u_h$ the Galerkin solution in $V_h\subset V$.
With the energy norm $\|v\|_b=b(v,v)^{1/2}$ prove (i) $b(u-u_h,v_h)=0$ for all $v_h\in V_h$; (ii) $\|u-u_h\|_b\le\|u-v_h\|_b$ for all $v_h\in V_h$;
(iii) $\|u-u_h\|_V\le\sqrt{M/\gamma}\,\inf_{v_h\in V_h}\|u-v_h\|_V$.

> *your proofs*

## Q2 — MMS for a variable-coefficient problem (35 pts)
$$-\nabla\cdot(k\nabla u)+c\,u=f\ \text{ in }\Omega=(0,1)^2,\qquad k\,\partial_nu=0\ \text{ on }\partial\Omega,\qquad k=1+x^2,\quad c=1+y .$$

**(a) (15)** Choose a manufactured solution $\tilde u$ that satisfies the boundary condition (justify), compute $f$ with UFL, and solve with
**your own assembly + KSP** (`fem.petsc.assemble_matrix`, `assemble_vector`, `ghostUpdate`, `PETSc.KSP` with LU — not `LinearProblem`).
Check the converged reason. Run with `-np 2`. Tabulate $L^2$ and $H^1$ errors and rates for P1 and P2, $N=8,16,32,64$, and plot them.

**(b) (10)** With $\|v\|_b^2=\int k|\nabla v|^2+cv^2$ compare $\|\tilde u-u_h\|_b$ with $\|\tilde u-\mathscr I_h\tilde u\|_b$ for P1 and P2, $N=8,16,32$.
Which is smaller, always? Which result of Q1 explains it?

**(c) (10)** P1, $N=32,64,128,256$: number of CG iterations (rtol $10^{-10}$) with Jacobi and with GAMG preconditioning, and the converged reason.
How does the Jacobi count grow with $N$, and why (hint: condition number of $K$)? Do the errors depend on the solver?

In [ ]:
%%fenicsx -np 2
# TODO: (a) manufactured solution, assembly + KSP, rates for P1 and P2; save the errors to /tmp/fnx/hw5_q2.npz

In [ ]:
# TODO: plot the errors from /tmp/fnx/hw5_q2.npz (log-log, x-axis labelled 1/N)

In [ ]:
%%fenicsx -np 2
# TODO: (b) energy-norm errors of u_h and of the interpolant

In [ ]:
%%fenicsx -np 2
# TODO: (c) CG iterations with Jacobi and GAMG, P1, N = 32 ... 256

> *(a)–(c) your answers: choice of ũ, rates, comparison, iteration counts and explanation*

## Q3 — Coercivity constants and dual norms, numerically (20 pts)
Use dense matrices on one process (`fem.assemble_matrix(...).to_dense()` / `.to_scipy()`), P1.

**(a) (12)** Compute $\gamma_h=\lambda_{\min}(K_b,G)$, $N=4,8,16$, for
(i) $b$ of Q2 on $H^1$; (ii) $\int\nabla u\cdot\nabla v$ on $\{v=0$ on all of $\partial\Omega\}$;
(iii) $\int\nabla u\cdot\nabla v$ on $\{v=0$ on the side $x=0$ only$\}$; (iv) $\int\nabla u\cdot\nabla v$ on $H^1$.
Compare (i) with your $\gamma$ of Q1(a). For (ii) and (iii), predict the limits: show that $\gamma=\lambda_1/(1+\lambda_1)$ where $\lambda_1$ is the
smallest eigenvalue of $-\Delta$ with the corresponding boundary conditions, and find $\lambda_1$.

**(b) (8)** Dual norm of point evaluation $L(v)=v(x_0)$ on $V_h$ with the $H^1$ norm: $\|L\|_{V_h'}=\sqrt{\mathbf e^TG^{-1}\mathbf e}$ where $\mathbf e$ is the
unit vector of the DOF at $x_0$ (a vertex). Compute it for $x_0=0.5$ on $(0,1)$ with $N=8,\dots,512$ cells, and for $x_0=(0.5,0.5)$ on the unit
square with $N=4,\dots,64$. Describe the two trends and relate them to the Sobolev embedding of Week 4 (lecture §A2).

In [ ]:
%%fenicsx -np 1
# TODO: (a) gamma_h for (i)–(iv)

In [ ]:
%%fenicsx -np 1
# TODO: (b) dual norm of point evaluation, 1D and 2D

> *(a), (b) your predictions and interpretation*

## Q4 — LLM-generated MMS study with planted bugs (20 pts)
The code below was produced by an LLM for the prompt *"MMS convergence study for $-\Delta u+u=f$ with $\partial u/\partial n=0$, P1 and P2, run
with `mpirun -np 2`"*. It contains **at least six** mistakes (API, mathematics, verification and parallel).

**(a) (12)** Fix it in the next code cell, so that it runs and reports correct rates.
**(b) (8)** Table: for each mistake give the line, what was wrong, how you detected it (error message, rate, `-np 1` vs `-np 2`, ...), and the fix.

*The first cell is meant to fail. The real error is the last traceback (printed once per rank). The lines
`Exception ignored ... LinearProblem.__del__ ... no attribute '_solver'` above it are a side effect of the failed constructor, not a separate mistake.*

In [ ]:
%%fenicsx -np 2
# EXPECTED-FAIL — the code as generated (do not edit this cell)
from mpi4py import MPI
import numpy as np
import ufl
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem

def solve_mms(N, k):
    msh = mesh.create_unit_square(MPI.COMM_WORLD, N, N)
    V = fem.functionspace(msh, ("Lagrange", k))
    u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    x = ufl.SpatialCoordinate(msh)
    u_ex = ufl.cos(ufl.pi * x[0]) * ufl.sin(ufl.pi * x[1])
    f = (ufl.pi**2 + 1) * u_ex                    # -Δu + u, differentiated by hand
    a = ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx + u * v * ufl.dx
    L = f * v * ufl.dx
    problem = LinearProblem(a, L, petsc_options={"ksp_type": "cg", "pc_type": "none", "ksp_max_it": 50})
    uh = problem.solve()
    u_ref = fem.Function(V)
    u_ref.interpolate(lambda x: np.cos(np.pi * x[0]) * np.sin(np.pi * x[1]))
    e = uh - u_ref
    L2 = np.sqrt(fem.assemble_scalar(fem.form(e * e * ufl.dx)))
    H1 = np.sqrt(fem.assemble_scalar(fem.form(ufl.inner(ufl.grad(e), ufl.grad(e)) * ufl.dx)))
    return L2, H1

Ns = np.array([8, 16, 32, 64])
for k in (1, 2):
    E = np.array([solve_mms(N, k) for N in Ns])
    rates = np.log(E[1:] / E[:-1]) / np.log(Ns[1:, None] / Ns[:-1, None])
    print(f"P{k} errors:\n{E}\nrates:\n{rates}")

In [ ]:
%%fenicsx -np 2
# TODO: corrected code

**Effect of each mistake alone** (a diagnostic cell: the corrected code with one mistake switched back on, P1, $N=8,16,32$).

In [ ]:
%%fenicsx -np 2
# (optional) your own diagnostics

> *(b) your table*

| # | line | mistake | how detected | fix |
|---|---|---|---|---|
| 1 | | | | |

## 🤖 AI-usage log

*Required in every assignment and project. Using an LLM is encouraged and never penalized — an empty or vague log is.*

| # | Tool / model | What I asked (key prompt, shortened) | What it got wrong or missed | How I detected it | How I fixed it |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

**Code provenance** — which cells/functions were mostly generated, which did you write or substantially change?

> …

**Verification** — how do you know the final result is correct (MMS rates, benchmark value, conservation check, parallel = serial)?

> …

**Reflection (2–3 sentences)** — what would you do differently when prompting next time?

> …